This notebook demonstrates running the consistent-agents framework on the SWEBench Benchmark with some example runs.

### What is SWEBench?

**SWEBench** is a benchmark designed to evaluate LLMs' ability to read, understand, and fix real-world bugs in open-source repositories.

Each instance contains:
- A natural-language bug report or issue description
- The repository name and base commit
- A patch that resolves the issue (the "oracle")
- A set of tests that verify correctness

**Metric**: Accuracy is defined as the fraction of correctly fixed tasks verified via the benchmark's test harness.

### Consistency Evaluation

To run the consistency evaluation of an agent using the framework, we need to define a config file typically placed in the `src/consistent_agents/config` directory. For experiments with the SWEBench benchmark, we used `src/consistent_agents/config/swebench.yaml`.

Please check out the config file—it will give you a better sense of what models, agents, and perturbations we used for this trial. We used the `LLMBackTranslationPerturbation` perturbation, which perturbs the issue description of a SWEBench sample written in English by first converting it to a non-English language and then translating it back to English. We believe that during back-translation, some variation is incurred in the text. This is a very simple perturbation method but good for initial testing.

We have already run the evaluation and the results are in `outputs/swebench-2025-11-18::20-16-05`. To run the evaluation yourself, you can use the following command:
```bash
pip install -e .
python3 -m src.consistent_agents.eval src/consistent_agents/config/swebench.yaml
```

In [8]:
import os 
import sys
import json
from pprint import pprint
from pathlib import Path

from IPython.display import display, HTML

In [9]:
def load_json(file_path):
    with open(file_path, "r") as file:
        dct = json.load(file)
    return dct

In [10]:
REPO_ROOT = Path.cwd().parent
RESULT_FILE_PATH = f"{REPO_ROOT}/outputs/swebench-2025-12-04::13-53-59/result.json"
TRAJECTORY_FILE_PATH = f"{REPO_ROOT}/outputs/swebench-2025-12-04::13-53-59/trajectory.json"

In [11]:
result = load_json(RESULT_FILE_PATH)
print("Accuracy:", result["accuracy"])
print("Consistency:", result["consistency"])

Accuracy: 0.0
Consistency: 0.0


In [12]:
def pretty_print_trajectory(trajectory):
    html_output = "<div style='font-family: Arial, sans-serif;'>"
    
    for i, turn in enumerate(trajectory):
        role = turn["role"]
        content = turn.get("content", "")
        
        # Handle different content types
        if isinstance(content, dict):
            content = json.dumps(content, indent=2)
        elif isinstance(content, list):
            content = json.dumps(content, indent=2)
        
        # Escape HTML in content
        content = str(content).replace("<", "&lt;").replace(">", "&gt;")
        
        if role == "system":
            color, bg_color, emoji = "#666", "#f5f5f5", "⚙️"
        elif role == "user":
            color, bg_color, emoji = "#1976D2", "#E3F2FD", "👤"
        elif role == "assistant":
            color, bg_color, emoji = "#7B1FA2", "#F3E5F5", "🤖"
        elif role == "tool" or role == "function":
            color, bg_color, emoji = "#F57C00", "#FFF3E0", "🔧"
        elif role == "error":
            color, bg_color, emoji = "#D32F2F", "#FFEBEE", "⚠️"
        else:
            color, bg_color, emoji = "#757575", "#FAFAFA", "💬"
        
        html_output += f"""
        <div style="background-color: {bg_color}; 
                    border-left: 5px solid {color}; 
                    padding: 15px; 
                    margin: 12px 0; 
                    border-radius: 8px;
                    box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
            <div style="display: flex; align-items: center; margin-bottom: 8px;">
                <span style="font-size: 1.2em; margin-right: 8px;">{emoji}</span>
                <strong style="color: {color}; font-size: 1.1em;">{role.upper()}</strong>
                <span style="color: #666; font-size: 0.9em; margin-left: 10px;">Turn {i+1}</span>
            </div>
            <pre style="margin: 0; 
                       white-space: pre-wrap; 
                       word-wrap: break-word;
                       font-family: 'Courier New', monospace; 
                       background-color: white; 
                       padding: 10px; 
                       border-radius: 4px;
                       font-size: 0.95em;
                       line-height: 1.5;
                       color: #000;
                       overflow-x: auto;">{content}</pre>
        </div>
        """
    
    html_output += "</div>"
    display(HTML(html_output))

In [13]:
trajectory = load_json(TRAJECTORY_FILE_PATH)

In [14]:
pretty_print_trajectory(trajectory["entries"][0]["messages"])

In [15]:
pretty_print_trajectory(trajectory["entries"][1]["messages"])